In [ ]:
## Set environment variables
import os
NEO4J_URI=os.environ["NEO4J_URI"]
NEO4J_USERNAME=os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD=os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE=os.environ["NEO4J_DATABASE"]
ACADEMIC_API_KEY=os.environ["ACADEMIC_API_KEY"]
GPT_API_KEY=os.environ["GPT_API_KEY"]

In [5]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
    max_connection_lifetime=3600 
)

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run("RETURN 'connected' AS status")
    print(result.single())

# driver.close()


<Record status='connected'>


In [6]:
# from langchain_community.graphs import Neo4jGraph
from langchain_neo4j import Neo4jGraph
graph=Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE, 
    sanitize=True # Cypher queries generated are safe to execute
)

In [7]:
graph

In [137]:
from langchain_openai import ChatOpenAI

CYPHER_MODEL="gpt-oss-120b"
QA_MODEL="gpt-oss-120b"

cypher_llm = ChatOpenAI(
    api_key=ACADEMIC_API_KEY,  
    base_url="https://chat-ai.academiccloud.de/v1",
    model=CYPHER_MODEL,
    temperature=0.2
)

qa_llm=ChatOpenAI(
    api_key=ACADEMIC_API_KEY,  
    base_url="https://chat-ai.academiccloud.de/v1",
    model=QA_MODEL,
    temperature=0.2
)

# llm.invoke("How tall is the Eiffel tower?")



In [ ]:
# from langchain_groq import ChatGroq
# groq_api_key= os.environ["GROQ_API_KEY"]
# qa_llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant")
# cypher_llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant")

### Load a Preprocess Data from Hugging Face

In [ ]:
from datasets import load_dataset
from langchain_core.documents import Document

# Load dataset
dataset = load_dataset("galileo-ai/ragbench", "hotpotqa", split="train")

# Check split name
split_name = list(dataset.keys())[0]  # usually 'train'
print(f"Using split: {split_name}")

# Take 0.1% of the data
subset = dataset[split_name].select(range(int(0.0001 * len(dataset[split_name]))))
subset = subset.filter(lambda row: row["type"] == "bridge")  # only bridge questions


def extract_title_passages(context_obj, min_tokens=20):
    """
    context_obj schema:
    {
      "title": [t1, t2, ...],
      "sentences": [[s11, s12, ...], [s21, s22, ...], ...]
    }
    Returns list of (title, passage_text)
    """
    titles = context_obj["title"]
    sentence_groups = context_obj["sentences"]

    assert len(titles) == len(sentence_groups), "Title–sentence alignment broken"

    passages = []
    for title, sentences in zip(titles, sentence_groups):
        text = " ".join(s.strip() for s in sentences if s and s.strip())
        if len(text.split()) >= min_tokens:
            passages.append((title, text))
    return passages


# Build Documents from context (NOT QA pairs)
documents = []
doc_id = 0

for row in subset:
    title_passages = extract_title_passages(row["context"])

    for title, passage_text in title_passages:
        documents.append(
            Document(
                page_content=passage_text,
                metadata={
                    "doc_id": doc_id,
                    "example_id": row["id"],
                    "title": title,
                }
            )
        )
        doc_id += 1

print(f"Total passages indexed: {len(documents)}")
print(documents[:2])

Using split: train
Total passages indexed: 67
[Document(metadata={'doc_id': 0, 'example_id': '5a879ab05542996e4f30887e', 'title': 'Ritz-Carlton Jakarta'}, page_content='The Ritz-Carlton Jakarta is a hotel and skyscraper in Jakarta, Indonesia and 14th Tallest building in Jakarta. It is located in city center of Jakarta, near Mega Kuningan, adjacent to the sister JW Marriott Hotel. It is operated by The Ritz-Carlton Hotel Company. The complex has two towers that comprises a hotel and the Airlangga Apartment respectively. The hotel was opened in 2005.'), Document(metadata={'doc_id': 1, 'example_id': '5a879ab05542996e4f30887e', 'title': 'Oberoi family'}, page_content='The Oberoi family is an Indian family that is famous for its involvement in hotels, namely through The Oberoi Group.')]


In [37]:
for i in range(len(documents)):
    doc = documents[i]
    print(f"\n--- Document {i} ---")
    print("example_id:", doc.metadata.get("example_id"))
    print("title:", doc.metadata.get("title"))
    print("content:", doc.page_content)  


--- Document 0 ---
example_id: 5a879ab05542996e4f30887e
title: Ritz-Carlton Jakarta
content: The Ritz-Carlton Jakarta is a hotel and skyscraper in Jakarta, Indonesia and 14th Tallest building in Jakarta. It is located in city center of Jakarta, near Mega Kuningan, adjacent to the sister JW Marriott Hotel. It is operated by The Ritz-Carlton Hotel Company. The complex has two towers that comprises a hotel and the Airlangga Apartment respectively. The hotel was opened in 2005.

--- Document 1 ---
example_id: 5a879ab05542996e4f30887e
title: Oberoi family
content: The Oberoi family is an Indian family that is famous for its involvement in hotels, namely through The Oberoi Group.

--- Document 2 ---
example_id: 5a879ab05542996e4f30887e
title: Ishqbaaaz
content: Ishqbaaaz (English: "Lovers") is an Indian drama television series which is broadcast on Star Plus. It premiered on 27 June 2016 and airs Mon-Fri 10-11pm IST. Nakuul Mehta, Kunal Jaisingh and Leenesh Mattoo respectively portray Shiva

In [38]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
llm_transformer=LLMGraphTransformer(llm=qa_llm)

In [39]:
graph_documents=llm_transformer.convert_to_graph_documents(documents) #convert raw text to structured knowledge--> nodes + relationships

In [9]:
graph_documents

NameError: name 'graph_documents' is not defined

In [10]:
graph_documents[0].nodes

NameError: name 'graph_documents' is not defined

In [42]:
graph_documents[0].relationships

[Relationship(source=Node(id='Ritz-Carlton Jakarta', type='Building', properties={}), target=Node(id='Jakarta', type='City', properties={}), type='LOCATED_IN', properties={}),
 Relationship(source=Node(id='Jakarta', type='City', properties={}), target=Node(id='Indonesia', type='Country', properties={}), type='LOCATED_IN', properties={}),
 Relationship(source=Node(id='Ritz-Carlton Jakarta', type='Building', properties={}), target=Node(id='Hotel', type='Concept', properties={}), type='TYPE', properties={}),
 Relationship(source=Node(id='Ritz-Carlton Jakarta', type='Building', properties={}), target=Node(id='Skyscraper', type='Concept', properties={}), type='TYPE', properties={}),
 Relationship(source=Node(id='Ritz-Carlton Jakarta', type='Building', properties={}), target=Node(id='Jakarta', type='City', properties={}), type='RANKED_AS_14TH_TALLEST_IN', properties={}),
 Relationship(source=Node(id='Ritz-Carlton Jakarta', type='Building', properties={}), target=Node(id='Mega Kuningan', type

In [ ]:
# # Convert data to CSV file, keeping only instruction and response
# import pandas as pd

# # Create DataFrame from subset
# df = pd.DataFrame(subset)

# # Keep only the two columns
# df = df[['instruction', 'response']]
# #
# # Save to CSV
# df.to_csv("data/bitext.csv", index=False)

In [ ]:
# from neo4j import GraphDatabase
# import pandas as pd

# # Connect to Neo4j
# uri = "bolt://localhost:7687"
# driver = GraphDatabase.driver(uri, auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]))

# # Load CSV in Python
# df = pd.read_csv("data/bitext.csv")

# # Insert into Neo4j
# with driver.session() as session:
#     for _, row in df.iterrows():
#         session.run(
#             """
#             MERGE (q:Instruction {text: $instruction})
#             MERGE (r:Response {text: $response})
#             MERGE (q)-[:HAS_RESPONSE]->(r)
#             """,
#             instruction=row['instruction'],
#             response=row['response']
#         )

# driver.close()
# print("Data imported into Neo4j")


In [60]:
# // Load the CSV file
# LOAD CSV WITH HEADERS FROM 'data/bitext.csv' AS row

# // Create an Instruction node
# MERGE (q:Instruction {text: row.instruction})

# // Create a Response node
# MERGE (r:Response {text: row.response})

# // Link Instruction to Response
# MERGE (q)-[:HAS_RESPONSE]->(r);


In [43]:
graph.add_graph_documents(graph_documents)

In [44]:
graph

In [11]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Building {id: STRING}
City {id: STRING}
Country {id: STRING}
Location {id: STRING}
Hotel {id: STRING}
Organization {id: STRING}
Apartment {id: STRING}
Date {id: STRING}
Concept {id: STRING}
Family {id: STRING}
Television_series {id: STRING}
Network {id: STRING}
Person {id: STRING}
Character {id: STRING}
Time_slot {id: STRING}
Event {id: STRING}
Monetary {id: STRING}
Group {id: STRING}
Brand {id: STRING}
State {id: STRING}
Product {id: STRING}
Address {id: STRING}
Material {id: STRING}
Year {id: STRING}
Episode {id: STRING}
Comic {id: STRING}
Fictional_character {id: STRING}
Tv_show {id: STRING}
Short {id: STRING}
Series {id: STRING}
Comic strip {id: STRING}
Word {id: STRING}
Alias {id: STRING}
Platform {id: STRING}
Film {id: STRING}
Television_show {id: STRING}
Game {id: STRING}
Album {id: STRING}
Podcast {id: STRING}
String {id: STRING}
Newspaper {id: STRING}
Cartoon_strip {id: STRING}
Company {id: STRING}
Article {id: STRING}
Show {id: STRING}
Book {id: STRING}
Tv_se

In [138]:
from langchain_core.prompts import PromptTemplate

CYPHER_GENERATION_TEMPLATE = """
You are an expert Neo4j Cypher generator for HotpotQA-style multi-hop QA.

TASK:
Generate ONE Cypher query that retrieves ALL properties required to answer the question fully.

STRICT RULES:
- Output ONLY executable Cypher.
- Do NOT include explanations, reasoning, markdown, or comments.
- Do NOT return nodes or maps (no RETURN t, a).
- Always RETURN scalar properties with aliases.
- Use only labels, relationships, and properties in the schema.
- Do NOT use OPTIONAL MATCH unless the relationship may not exist.
- No CREATE, MERGE, DELETE, SET.
- The query must return enough information to reconstruct the full factual answer.

ANSWER COMPLETENESS RULES:
- For multi-part questions, retrieve each fact explicitly.

SCHEMA:
{schema}

QUESTION:
{question}
"""
prompt = PromptTemplate(
    input_variables=["schema", "question"],
    template=CYPHER_GENERATION_TEMPLATE,
)

In [139]:
# GraphCypherQAChain comes from langchain_neo4j; use the existing qa_llm and cypher_llm
from langchain_neo4j import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
	llm=qa_llm,
	cypher_llm=cypher_llm,
	graph=graph,
    prompt=prompt,
	verbose=True,
    allow_dangerous_requests=True
)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_neo4j.graphs.neo4j_graph.Neo4jGraph object at 0x0000021CB11B3230>, cypher_generation_chain=PromptTemplate(input_variables=['examples', 'question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nExamples (optional):\n{examples}\n\nThe question is:\n{question}')
| RunnableBinding(bound=ChatOpenAI(profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000021CB31B87D0>, async_client=<openai.resources.chat.completio

## Q1

In [14]:

response = chain.invoke({"query":"where is Ritz-Carlton Jakarta located?"})
response





> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (b:Building {id: "Ritz-Carlton Jakarta"})-[:LOCATED_IN]->(loc) RETURN loc.id AS location;
Full Context:
[{'location': 'Jakarta'}]

> Finished chain.


{'query': 'where is Ritz-Carlton Jakarta located?',
 'result': 'Ritz‑Carlton Jakarta is located in Jakarta.'}

In [43]:
response=chain.invoke({"query":"when is Hotel Bond build"})
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (h:Hotel {id: "Hotel Bond"})-[:BUILT_IN]->(d)
RETURN d.id AS buildDate
Full Context:
[{'buildDate': '1913'}, {'buildDate': '1921'}]

> Finished chain.


{'query': 'when is Hotel Bond build',
 'result': 'Hotel\u202fBond was built in\u202f1913\u202fand\u202f1921.'}

In [111]:
response=chain.invoke({"query":" What is The Oberoi Group and where is its head office located?"})
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (o:Organization {id: "The Oberoi Group"})
OPTIONAL MATCH (o)-[:HEADQUARTER_IN]->(loc:Location)
RETURN o.id AS organization, loc.id AS head_office_location;
Full Context:
[{'organization': 'The Oberoi Group', 'head_office_location': None}]

> Finished chain.


{'query': ' What is The Oberoi Group and where is its head office located?',
 'result': 'The Oberoi Group is an organization, and its head office location is not specified.'}

In [112]:
response=chain.invoke({"query":"based upon what is Batibot probuced?"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (b:Television_series {id: 'Batibot'})-[:BASED_ON]->(source) RETURN source;
Full Context:
[{'source': {'id': 'Sesame Street'}}]

> Finished chain.


{'query': 'based upon what is Batibot probuced?',
 'result': 'Batibot is produced based upon Sesame\u202fStreet.'}

In [102]:
response=chain.invoke({"query":"What is Wolfblood and who is its target audience"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (ts:Television_series {id: 'Wolfblood'})-[:TARGET_AUDIENCE]->(aud:Audience)
RETURN ts.id AS series, aud.id AS target_audience;
Full Context:
[{'series': 'Wolfblood', 'target_audience': 'Young Adult Audience'}]

> Finished chain.


{'query': 'What is Wolfblood and who is its target audience',
 'result': 'Wolfblood is a series, and its target audience is the Young Adult Audience.'}

In [143]:
response=chain.invoke({"query":"What is Future Fibre Technologies and where is it based?"})
response




> Entering new GraphCypherQAChain chain...


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. ':HEADQUARTER_IN|:BASED_IN|:LOCATED_IN|:LOCATED_AT' is deprecated. It is replaced by ':HEADQUARTER_IN|BASED_IN|LOCATED_IN|LOCATED_AT'.", position=<SummaryInputPosition line=2, column=60, offset=118>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 118, 'line': 2, 'column': 60}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (org:Organization {id: "Future Fibre Technologies"})\nOPTIONAL MATCH (org)-[:HEADQUARTER_IN|:BASED_IN|:LOCATED_IN|:LOCATED_AT]->(loc)\nRETURN org, labels(org) AS type, loc.id AS location_id, labels(loc) AS location_type;'


Generated Cypher:
MATCH (org:Organization {id: "Future Fibre Technologies"})
OPTIONAL MATCH (org)-[:HEADQUARTER_IN|:BASED_IN|:LOCATED_IN|:LOCATED_AT]->(loc)
RETURN org, labels(org) AS type, loc.id AS location_id, labels(loc) AS location_type;
Full Context:
[{'org': {'id': 'Future Fibre Technologies'}, 'type': ['Organization'], 'location_id': 'Melbourne', 'location_type': ['City']}]

> Finished chain.


{'query': 'What is Future Fibre Technologies and where is it based?',
 'result': 'Future Fibre Technologies is an organization based in Melbourne.'}

## Q2

In [31]:
response=chain.invoke({"query":"How  to place an order"})
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ASK]->(o:Order)
RETURN p.id AS personId, o.id AS orderId;
Full Context:
[{'personId': 'Customer', 'orderId': '{{Order Number}}'}]

> Finished chain.


{'query': 'How  to place an order',
 'result': 'The customer places an order by providing the order number\u202f{{Order Number}}.'}

In [31]:
response=chain.invoke({"query":"How  to place an order"})
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:NAVIGATE|INITIATE]->(f:Feature)-[:MANAGES]->(o:Order)
RETURN p.id AS personId, f.id AS featureId, o.id AS orderId;
Full Context:
[]

> Finished chain.


{'query': 'How  to place an order', 'result': 'I don’t know the answer.'}

In [32]:
response=chain.invoke({"query":"i need assisstance placing an order"})
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (o:Order)<-[:SUPPORT_FOR]-(w:Website)
OPTIONAL MATCH (p:Person)-[:CONTACT_SUPPORT]->(w)
RETURN o.id AS orderId, w.id AS websiteId, collect(p.id) AS supportPersons;
Full Context:
[{'orderId': '{{Order Number}}', 'websiteId': '{{Website Url}}', 'supportPersons': ['User']}]

> Finished chain.


{'query': 'i need assisstance placing an order',
 'result': 'You can place your order on {{Website Url}} using order number {{Order Number}}. If you need help, you can contact User for assistance.'}

In [36]:
response=chain.invoke({"query":"i need some help to place an order"})
response




> Entering new GraphCypherQAChain chain...


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. ':MANAGE|:APPLY' is deprecated. It is replaced by ':MANAGE|APPLY'.", position=<SummaryInputPosition line=2, column=35, offset=99>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 99, 'line': 2, 'column': 35}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (w:Website)-[:PROVIDES]->(f:Feature)-[:MANAGES]->(o:Order)\nOPTIONAL MATCH (a:Action)-[:MANAGE|:APPLY]->(o)\nRETURN w.id AS website, f.id AS feature, a.id AS action\nLIMIT 10'


Generated Cypher:
MATCH (w:Website)-[:PROVIDES]->(f:Feature)-[:MANAGES]->(o:Order)
OPTIONAL MATCH (a:Action)-[:MANAGE|:APPLY]->(o)
RETURN w.id AS website, f.id AS feature, a.id AS action
LIMIT 10
Full Context:
[{'website': 'Online Company Portal Info', 'feature': 'Online Order Interaction', 'action': None}]

> Finished chain.


{'query': 'i need some help to place an order',
 'result': 'You can place your order through the Online Company Portal, which provides an online order interaction feature.'}

In [47]:
response=chain.invoke({"query":" How can i place an order"})
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {id: "some_person_id"})-[:INITIATE]->(f:Feature)-[:MANAGES]->(o:Order)
RETURN o.id AS orderId
UNION
MATCH (p:Person {id: "some_person_id"})-[:INITIATE]->(a:Action)-[:MANAGE]->(o:Order)
RETURN o.id AS orderId
Full Context:
[]

> Finished chain.


{'query': ' How can i place an order', 'result': 'I don’t know the answer.'}

## Q3

In [46]:
response=chain.invoke({"query":" I am waiting for an refund of {{Currency Symbol}}{{Refund Amount}}"})
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:HAS_PROBLEM_WITH]->(o:Order)
RETURN p.id AS personId, o.id AS orderId
Full Context:
[{'personId': 'User', 'orderId': '{{Order Number}}'}]

> Finished chain.


{'query': ' I am waiting for an refund of {{Currency Symbol}}{{Refund Amount}}',
 'result': 'I don’t know the answer.'}

In [51]:
response=chain.invoke({"query":" want assistance to know if there are any news on my rebate"})
response





> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Message)
WHERE m.id CONTAINS 'rebate'
RETURN m;
Full Context:
[]

> Finished chain.


{'query': ' want assistance to know if there are any news on my rebate',
 'result': 'I’m sorry, but I don’t have the information needed to answer that.'}

In [52]:
response=chain.invoke({"query":" see the status of the reimbursement"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (o:Order) RETURN o.id;
Full Context:
[{'o.id': 'Order'}, {'o.id': '{{Order Number}}'}, {'o.id': 'Order Number'}, {'o.id': 'Order {{Order Number}}'}]

> Finished chain.


{'query': ' see the status of the reimbursement',
 'result': 'I don’t know the answer.'}

### BERTScore

## BERTScore

In [80]:
import json
from tqdm import tqdm
from bert_score import score

def evaluate_graph_rag_with_bertscore(chain, subset, max_examples=None):
    """
    Runs GraphCypherQAChain on test examples and computes BERTScore.
    
    Args:
        chain: GraphCypherQAChain instance
        subset: HuggingFace dataset subset with fields ['instruction', 'response']
        max_examples: int | None (limit evaluation size for faster runs)
        
    Returns:
        avg_precision, avg_recall, avg_f1
    """

    references = []
    predictions = []

    examples = subset
    if max_examples is not None:
        examples = subset.select(range(min(max_examples, len(subset))))

    for row in tqdm(examples, desc="Evaluating Graph RAG"):
        question = row["instruction"]
        reference = row["response"]

        try:
            result = chain.invoke({"query": question})

            # LangChain GraphCypherQAChain returns dict like:
            # {"query": "...", "result": "..."}
            predicted_answer = result.get("result", "")

        except Exception as e:
            print(f"[WARN] Query failed: {question[:80]}...")
            print(e)
            predicted_answer = ""

        predictions.append(predicted_answer)
        references.append(reference)

    # Compute BERTScore
    P, R, F1 = score(
        predictions,
        references,
        lang="en",
        model_type="distilroberta-base",
        batch_size=16,
        rescale_with_baseline=True
    )

    avg_precision = P.mean().item()
    avg_recall = R.mean().item()
    avg_f1 = F1.mean().item()

    print(f"\n[RESULT] BERTScore")
    print(f"Precision: {avg_precision:.4f}")
    print(f"Recall   : {avg_recall:.4f}")
    print(f"F1       : {avg_f1:.4f}")

    return avg_precision, avg_recall, avg_f1


In [81]:
# Evaluate on the same subset inserted into Neo4j
evaluate_graph_rag_with_bertscore(
    chain=chain,
    subset=subset,
    max_examples=30  
)

Evaluating Graph RAG:   0%|          | 0/26 [00:00<?, ?it/s]



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (o:Order {id: "{{Order Number}}"}),(p:Person)-[:CONFIRM|CANCEL|LOCATE_IN_SECTION|CLICK|GO_TO|CONTACT]->(f:Feature)-[:PROVIDES]->(s:Service)-[:OFFER]->(o) RETURN p.id AS user_id, f.id AS feature_id;
Full Context:
[]


Evaluating Graph RAG:   4%|▍         | 1/26 [00:01<00:31,  1.28s/it]


> Finished chain.


> Entering new GraphCypherQAChain chain...


Evaluating Graph RAG:   8%|▊         | 2/26 [00:07<01:35,  4.00s/it]

...


> Finished chain.


> Entering new GraphCypherQAChain chain...


Evaluating Graph RAG: 100%|██████████| 26/26 [12:54<00:00, 29.78s/it]

Generated Cypher:
MATCH (p:Person)-[:INITIATE_CANCELLATION]->(o:Order {id: $OrderNumber}) RETURN p
[WARN] Query failed: i want assistance cancelling order {{Order Number}}...
{neo4j_code: Neo.ClientError.Statement.ParameterMissing} {message: Expected parameter(s): OrderNumber} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 284.38it/s, Materializing param=pooler.dense.weight]                             
RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AttributeError: RobertaTokenizer has no attribute build_inputs_with_special_tokens